In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import json
import os
import cv2
import random
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

In [5]:
# 음...이건 그냥 코드 돌릴때 일정하게 나올라고 설정한 거임.
# 1. 파이썬 내장 random 시드 고정
random.seed(42)

# 2. 넘파이 시드 고정
np.random.seed(42)

# 3. TensorFlow 시드 고정
tf.random.set_seed(42)

# 4. 환경 변수 (특히 GPU 연산의 비결정성 제어)
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_DETERMINISTIC_OPS"] = "1"  # TF 연산을 결정적으로

In [21]:
plant_name = "부레옥잠"

In [22]:
# ???_dataset_classification
# (1) 드라이브 → Colab으로 복사
zip_path = f"/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(classification)/{plant_name}_dataset_classification.zip"

target_zip = f"/content/{plant_name}_dataset_classification.zip"

extract_dir = f"/content/{plant_name}_dataset_classification"

!cp "$zip_path" "$target_zip"
!unzip -q "$target_zip" -d "$extract_dir"
!rm "$target_zip"

In [23]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    roc_auc_score,
    accuracy_score,
    precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

# =========================================================
# 1. 사용자 설정 영역
# =========================================================

# 학습 완료된 모델 파일 경로
MODEL_PATH = f"/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/mobilenetv3_log/{plant_name}_mobilenetv3_large_best(final).keras"

# 테스트 데이터셋 폴더 경로
# 예: /content/몬스테라_dataset_classification/test
TEST_DIR = f"/content/{plant_name}_dataset_classification/test"

# 저장할 PDF 파일 경로
PDF_SAVE_PATH = f"/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/과습 여부 분류 성능 지표 모음/{plant_name}_test_performance_report.pdf"

# 이미지 크기 / 배치 크기
IMG_SZ = (224, 224)
BATCH = 32

# =========================================================
# 2. 테스트 데이터 로더
# =========================================================

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_gen = test_datagen.flow_from_directory(
    directory=TEST_DIR,
    target_size=IMG_SZ,
    batch_size=BATCH,
    class_mode="categorical",   # 학습 코드와 동일
    shuffle=False               # 평가 시 반드시 False
)

class_indices = test_gen.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
num_classes = len(class_names)

print("클래스 매핑:", class_indices)

# =========================================================
# 3. 모델 로드
# =========================================================

model = tf.keras.models.load_model(MODEL_PATH)

# =========================================================
# 4. 예측 수행
# =========================================================

# test_gen.classes : 정답 라벨(정수 인덱스)
y_true = test_gen.classes

# predict 결과: 각 클래스에 대한 확률
y_prob = model.predict(test_gen, verbose=1)

# 최종 예측 클래스
y_pred = np.argmax(y_prob, axis=1)

# one-hot / binarized 정답 생성 (ROC 계산용)
y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))

# =========================================================
# 5. 기본 성능 계산
# =========================================================

test_loss, test_acc = model.evaluate(test_gen, verbose=0)
acc = accuracy_score(y_true, y_pred)

cm = confusion_matrix(y_true, y_pred)

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_text = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4,
    zero_division=0
)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=np.arange(num_classes),
    zero_division=0
)

# =========================================================
# 6. ROC / AUC 계산 (다중분류)
# =========================================================

fpr = {}
tpr = {}
roc_auc = {}

for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# micro-average ROC
fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# macro-average ROC
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(num_classes)]))
mean_tpr = np.zeros_like(all_fpr)

for i in range(num_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

mean_tpr /= num_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# =========================================================
# 7. PDF 저장
# =========================================================

with PdfPages(PDF_SAVE_PATH) as pdf:

    # -----------------------------------------------------
    # Page 1: Summary
    # -----------------------------------------------------
    fig = plt.figure(figsize=(8.27, 11.69))  # A4 세로
    plt.axis("off")

    summary_lines = [
        "Test Performance Report",
        "",
        f"Model path: {MODEL_PATH}",
        f"Test directory: {TEST_DIR}",
        "",
        f"Number of test samples: {len(y_true)}",
        f"Class names: {class_names}",
        "",
        f"Test Loss     : {test_loss:.6f}",
        f"Test Accuracy : {test_acc:.6f}",
        f"Accuracy(sklearn): {acc:.6f}",
        "",
        f"ROC AUC Micro : {roc_auc['micro']:.6f}",
        f"ROC AUC Macro : {roc_auc['macro']:.6f}",
    ]

    plt.text(
        0.05, 0.95,
        "\n".join(summary_lines),
        va="top",
        ha="left",
        fontsize=12,
        family="monospace"
    )
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # -----------------------------------------------------
    # Page 2: Confusion Matrix
    # -----------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(num_classes),
        yticks=np.arange(num_classes),
        xticklabels=class_names,
        yticklabels=class_names,
        xlabel="Predicted Label",
        ylabel="True Label",
        title="Confusion Matrix"
    )

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    thresh = cm.max() / 2.0 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # -----------------------------------------------------
    # Page 3: ROC Curve
    # -----------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 6))

    for i in range(num_classes):
        ax.plot(
            fpr[i], tpr[i],
            lw=2,
            label=f"{class_names[i]} (AUC = {roc_auc[i]:.4f})"
        )

    ax.plot(
        fpr["micro"], tpr["micro"],
        linestyle="--",
        lw=2,
        label=f"micro-average (AUC = {roc_auc['micro']:.4f})"
    )

    ax.plot(
        fpr["macro"], tpr["macro"],
        linestyle=":",
        lw=2,
        label=f"macro-average (AUC = {roc_auc['macro']:.4f})"
    )

    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Multi-class ROC Curve")
    ax.legend(loc="lower right", fontsize=9)
    fig.tight_layout()

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # -----------------------------------------------------
    # Page 4: Classification Report (text)
    # -----------------------------------------------------
    fig = plt.figure(figsize=(8.27, 11.69))
    plt.axis("off")

    plt.text(
        0.03, 0.98,
        "Classification Report\n\n" + report_text,
        va="top",
        ha="left",
        fontsize=10,
        family="monospace"
    )

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # -----------------------------------------------------
    # Page 5: Per-class Metrics Table
    # -----------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.axis("off")

    table_data = []
    for i in range(num_classes):
        table_data.append([
            class_names[i],
            f"{precision[i]:.4f}",
            f"{recall[i]:.4f}",
            f"{f1[i]:.4f}",
            f"{support[i]}",
            f"{roc_auc[i]:.4f}"
        ])

    col_labels = ["Class", "Precision", "Recall", "F1-score", "Support", "ROC AUC"]

    table = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)

    ax.set_title("Per-class Performance Metrics", pad=20)
    fig.tight_layout()

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print(f"PDF 저장 완료: {PDF_SAVE_PATH}")

Found 3590 images belonging to 3 classes.
클래스 매핑: {'normal': 0, 'overwatered': 1, 'underwatered': 2}
113/113 ━━━━━━━━━━━━━━━━━━━━ 138s 1s/step
PDF 저장 완료: /content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/과습 여부 분류 성능 지표 모음/보스턴고사리_test_performance_report.pdf


# 인생은 한방

In [24]:
plants = ['관음죽', '금전수', '디펜바키아', '몬스테라', '벵갈고무나무', '보스턴고사리', '부레옥잠', '선인장', '스투키', '스파티필럼', '오렌지쟈스민', '올리브나무', '테이블야자', '호접란', '홍콩야자']

for plant_name in plants:
    # ???_dataset_classification
    # (1) 드라이브 → Colab으로 복사
    zip_path = f"/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(classification)/{plant_name}_dataset_classification.zip"

    target_zip = f"/content/{plant_name}_dataset_classification.zip"

    extract_dir = f"/content/{plant_name}_dataset_classification"

    !cp "$zip_path" "$target_zip"
    !unzip -q "$target_zip" -d "$extract_dir"
    !rm "$target_zip"

    import os
    import numpy as np
    import tensorflow as tf
    import matplotlib.pyplot as plt

    from matplotlib.backends.backend_pdf import PdfPages
    from sklearn.metrics import (
        confusion_matrix,
        classification_report,
        roc_curve,
        auc,
        roc_auc_score,
        accuracy_score,
        precision_recall_fscore_support
    )
    from sklearn.preprocessing import label_binarize
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

    # =========================================================
    # 1. 사용자 설정 영역
    # =========================================================

    # 학습 완료된 모델 파일 경로
    MODEL_PATH = f"/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/mobilenetv3_log/{plant_name}_mobilenetv3_large_best(final).keras"

    # 테스트 데이터셋 폴더 경로
    # 예: /content/몬스테라_dataset_classification/test
    TEST_DIR = f"/content/{plant_name}_dataset_classification/test"

    # 저장할 PDF 파일 경로
    PDF_SAVE_PATH = f"/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/과습 여부 분류 성능 지표 모음/{plant_name}_test_performance_report.pdf"

    # 이미지 크기 / 배치 크기
    IMG_SZ = (224, 224)
    BATCH = 32

    # =========================================================
    # 2. 테스트 데이터 로더
    # =========================================================

    test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

    test_gen = test_datagen.flow_from_directory(
        directory=TEST_DIR,
        target_size=IMG_SZ,
        batch_size=BATCH,
        class_mode="categorical",   # 학습 코드와 동일
        shuffle=False               # 평가 시 반드시 False
    )

    class_indices = test_gen.class_indices
    idx_to_class = {v: k for k, v in class_indices.items()}
    class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
    num_classes = len(class_names)

    print(f"{plant_name} - 클래스 매핑:", class_indices)

    # =========================================================
    # 3. 모델 로드
    # =========================================================

    model = tf.keras.models.load_model(MODEL_PATH)

    # =========================================================
    # 4. 예측 수행
    # =========================================================

    # test_gen.classes : 정답 라벨(정수 인덱스)
    y_true = test_gen.classes

    # predict 결과: 각 클래스에 대한 확률
    y_prob = model.predict(test_gen, verbose=1)

    # 최종 예측 클래스
    y_pred = np.argmax(y_prob, axis=1)

    # one-hot / binarized 정답 생성 (ROC 계산용)
    y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))

    # =========================================================
    # 5. 기본 성능 계산
    # =========================================================

    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    acc = accuracy_score(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(num_classes),
        zero_division=0
    )

    # =========================================================
    # 6. ROC / AUC 계산 (다중분류)
    # =========================================================

    fpr = {}
    tpr = {}
    roc_auc = {}

    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # micro-average ROC
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    # macro-average ROC
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(num_classes)]))
    mean_tpr = np.zeros_like(all_fpr)

    for i in range(num_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

    mean_tpr /= num_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    # =========================================================
    # 7. PDF 저장
    # =========================================================

    with PdfPages(PDF_SAVE_PATH) as pdf:

        # -----------------------------------------------------
        # Page 1: Summary
        # -----------------------------------------------------
        fig = plt.figure(figsize=(8.27, 11.69))  # A4 세로
        plt.axis("off")

        summary_lines = [
            "Test Performance Report",
            "",
            f"Model path: {MODEL_PATH}",
            f"Test directory: {TEST_DIR}",
            "",
            f"Number of test samples: {len(y_true)}",
            f"Class names: {class_names}",
            "",
            f"Test Loss     : {test_loss:.6f}",
            f"Test Accuracy : {test_acc:.6f}",
            f"Accuracy(sklearn): {acc:.6f}",
            "",
            f"ROC AUC Micro : {roc_auc['micro']:.6f}",
            f"ROC AUC Macro : {roc_auc['macro']:.6f}",
        ]

        plt.text(
            0.05, 0.95,
            "\n".join(summary_lines),
            va="top",
            ha="left",
            fontsize=12,
            family="monospace"
        )
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # -----------------------------------------------------
        # Page 2: Confusion Matrix
        # -----------------------------------------------------
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
        ax.figure.colorbar(im, ax=ax)

        ax.set(
            xticks=np.arange(num_classes),
            yticks=np.arange(num_classes),
            xticklabels=class_names,
            yticklabels=class_names,
            xlabel="Predicted Label",
            ylabel="True Label",
            title="Confusion Matrix"
        )

        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

        thresh = cm.max() / 2.0 if cm.max() > 0 else 0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(
                    j, i, format(cm[i, j], "d"),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black"
                )

        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # -----------------------------------------------------
        # Page 3: ROC Curve
        # -----------------------------------------------------
        fig, ax = plt.subplots(figsize=(8, 6))

        for i in range(num_classes):
            ax.plot(
                fpr[i], tpr[i],
                lw=2,
                label=f"{class_names[i]} (AUC = {roc_auc[i]:.4f})"
            )

        ax.plot(
            fpr["micro"], tpr["micro"],
            linestyle="--",
            lw=2,
            label=f"micro-average (AUC = {roc_auc['micro']:.4f})"
        )

        ax.plot(
            fpr["macro"], tpr["macro"],
            linestyle=":",
            lw=2,
            label=f"macro-average (AUC = {roc_auc['macro']:.4f})"
        )

        ax.plot([0, 1], [0, 1], "k--", lw=1)
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("Multi-class ROC Curve")
        ax.legend(loc="lower right", fontsize=9)
        fig.tight_layout()

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # -----------------------------------------------------
        # Page 4: Classification Report (text)
        # -----------------------------------------------------
        fig = plt.figure(figsize=(8.27, 11.69))
        plt.axis("off")

        plt.text(
            0.03, 0.98,
            "Classification Report\n\n" + report_text,
            va="top",
            ha="left",
            fontsize=10,
            family="monospace"
        )

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # -----------------------------------------------------
        # Page 5: Per-class Metrics Table
        # -----------------------------------------------------
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.axis("off")

        table_data = []
        for i in range(num_classes):
            table_data.append([
                class_names[i],
                f"{precision[i]:.4f}",
                f"{recall[i]:.4f}",
                f"{f1[i]:.4f}",
                f"{support[i]}",
                f"{roc_auc[i]:.4f}"
            ])

        col_labels = ["Class", "Precision", "Recall", "F1-score", "Support", "ROC AUC"]

        table = ax.table(
            cellText=table_data,
            colLabels=col_labels,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 2)

        ax.set_title("Per-class Performance Metrics", pad=20)
        fig.tight_layout()

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    print(f"PDF 저장 완료: {PDF_SAVE_PATH}")

Found 3593 images belonging to 3 classes.
클래스 매핑: {'normal': 0, 'overwatered': 1, 'underwatered': 2}
113/113 ━━━━━━━━━━━━━━━━━━━━ 142s 1s/step
PDF 저장 완료: /content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/과습 여부 분류 성능 지표 모음/부레옥잠_test_performance_report.pdf
Found 3462 images belonging to 3 classes.
클래스 매핑: {'normal': 0, 'overwatered': 1, 'underwatered': 2}
109/109 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step
PDF 저장 완료: /content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/과습 여부 분류 성능 지표 모음/선인장_test_performance_report.pdf
Found 3600 images belonging to 3 classes.
클래스 매핑: {'normal': 0, 'overwatered': 1, 'underwatered': 2}
113/113 ━━━━━━━━━━━━━━━━━━━━ 132s 1s/step
PDF 저장 완료: /content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/과습 여부 분류 성능 지표 모음/스투키_test_performance_report.pdf
Found 3600 images belonging to 3 classes.
클래스 매핑: {'normal': 0, 'overwatered': 1, 'underwatered': 2}
113/113 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step
PDF 저장 완료: /content/drive/My